In [2]:
%load_ext autoreload
%autoreload 2

from IPython.display import display
from src.utils import build_data_from_suffix

In [3]:
DATA = build_data_from_suffix("syntax", csv_dir="aligned")

In [11]:
from src.utils import save_str_ud_deprel_mismatches

out = save_str_ud_deprel_mismatches(DATA, "new", "атриб", "nmod", "obl")

# New:

### предик <-> nsubj:pass <-> nsubj

Это “ложный пассив на подлежащем” в предложениях с двумя предикациями (обычно координация/присоединение): в UD-графе корневым считается активный предикат (например, `опустил`), а пассивная предикация оформлена как `conj` (`разочарован`) с `aux:pass`; при этом конвертер, видимо, определяет `nsubj` vs `nsubj:pass` по наличию признака страдательности где-то “в окрестности” (в самом токене/его вершине/цепочке), и из-за этого подлежащее, которое синтаксически привязано к активному `root`, ошибочно получает `nsubj:pass` — пассивный признак относится не к главной вершине (head подлежащего), а к соседней пассивной клаузе, поэтому метка подлежащего переносится “не туда”.

### опред <-> acl <-> amod

Это группа ошибок “ложное `acl` вместо `amod`” при конвертации СинТагРус `опред`: конвертер решает `amod` vs `acl` по факту наличия у модификатора зависимых, и если находит хоть какие-то, переводит в `acl` (“распространенная группа/причастный оборот”), хотя в реальных примерах зависимые часто оказываются структурными (`conj` в рядах однородных определений или `parataxis` во вставках/уточнениях) и **не делают** определение распространенным в смысле UD-определения `acl`; в результате простые атрибуты, которые по правилу должны оставаться `amod`, систематически помечаются как `acl` только из‑за “нерасширяющих” зависимостей.

### аппоз <-> appos <-> flat:name

Это группа ошибок, где модель систематически предсказывает `flat:name`, а в UD-gold стоит `appos`, потому что конструкции вида “титул/родовой термин + имя” (например, “мосье Годар”, “пик Маттерхорн”) иногда ошибочно интерпретируются как многословное собственное имя; но по UD-определению это скорее аппозиция: второе имя (PROPN) непосредственно уточняет/называет первый именной узел (“мосье”, “пик”), поэтому корректная связь — `appos`, а `flat:name` должна применяться к собственным именам, состоящим из нескольких токенов внутри самой name-группы, а не к сочетанию “класс + имя”.

### разъяснит <-> appos <-> parataxis

Это группа ошибок, где модель переоценивает `parataxis` (вероятно, из‑за сильной ассоциации двоеточия/тире и “вставных кусков” с прямой речью и дискурсивным присоединением), тогда как в этих примерах после `:`/`—` идёт не самостоятельная клауза, а именной перечень/уточнение, то есть номинал конкретизирует номинал (“зрелище: пальто…”, “экипаж…: тракторист…”, “продукты: бутылки…”); поэтому по UD‑определению корректнее `appos`, а `parataxis` был бы уместен скорее при присоединении целого предложения/реплики, а не при номинальной расшифровке.

### атриб <-> nmod <-> obl 

Это группа ошибок, где в UD-gold стоит `nmod`, а модель предсказывает `obl`, потому что в данных встречаются конструкции, где именная группа (часто с предлогом: “для экономики”, “на один глаз”, “по отдельности”, “по определению”, “для них”) зависит не от существительного, а от предикативного/квази‑предикативного узла (прилагательного, глагола или даже детерминатива/местоимения) и фактически играет роль обстоятельственного/неядерного аргумента; по UD v2 такие зависимые должны размечаться как `obl`, тогда как `nmod` ограничен модификацией именных вершин, поэтому расхождение `атриб` → `nmod` (gold) ↔ `obl` (pred) можно интерпретировать как след старой практики/конвертации, где `nmod` ещё применяли при ADJ/VERB/ADV, и модель “исправляет” это в сторону современной нормы.